In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('affiliation-researchers-building-artificial-intelligence-systems-all.csv')

# years as rows and entities as columns
df_wide = df.pivot(index='Year', columns='Entity',
                   values='Annual number of AI systems by researcher affiliation')

full_years = range(df_wide.index.min(), df_wide.index.max() + 1)
df_wide = df_wide.reindex(full_years, fill_value=0)

print(df_wide.shape)
print(list(df_wide.columns))

(76, 4)
['Academia', 'Academia and industry collaboration', 'Industry', 'Other']


In [2]:
categories = ['Industry', 'Academia and industry collaboration', 'Academia', 'Other']
color_map = {
    'Industry': '#124E78',
    'Academia and industry collaboration': '#609836',
    'Academia': '#E36414',
    'Other': '#E8EB4E'
}

totals = df_wide[categories].sum(axis=1)
shares = df_wide[categories].div(totals, axis=0).fillna(0) * 100

print(totals.loc[2021])
print(shares.loc[2021].round(1))
print('Years with zero systems:', (totals == 0).sum())

custom = np.column_stack([df_wide[categories], shares, totals])
print(custom.shape)

79
Entity
Industry                               58.2
Academia and industry collaboration    25.3
Academia                               11.4
Other                                   5.1
Name: 2021, dtype: float64
Years with zero systems: 15
(76, 9)


In [3]:
def make_tooltip(current_category):

    # tooltip with the year from the x axis
    tooltip_lines = []
    tooltip_lines.append('<b>%{x}</b>')

    # go through each category from last to first
    category_numbers = range(len(categories))
    category_numbers = reversed(category_numbers)

    for index in category_numbers:

        # category name
        category_name = categories[index]

        # colour for this category
        category_colour = color_map[category_name]

        # make the coloured square shown next to the category
        colour_square = (
            '<span style="color:' + category_colour + '">■</span> '
        )

        #first four customdata columns contain the counts
        count_position = index

        #next four customdata columns contain the percentages
        percentage_position = index + 4

        #text for count and percentage
        category_text = (
            category_name
            + ': %{customdata['
            + str(count_position)
            + ']} (%{customdata['
            + str(percentage_position)
            + ']:.1f}%)'
        )

        # category that the mouse is currently on in bold
        if category_name == current_category:
            tooltip_lines.append(
                colour_square + '<b>' + category_text + '</b>'
            )

        #other categories grey
        else:
            tooltip_lines.append(
                colour_square
                + '<span style="color:#777777">'
                + category_text
                + '</span>'
            )

    # customdata column 8 contains the total
    tooltip_lines.append('Total: %{customdata[8]}')

    #put the tooltip on a new line
    tooltip = '<br>'.join(tooltip_lines)

    # remove Plotly trace name box
    tooltip = tooltip + '<extra></extra>'

    return tooltip

fig = go.Figure()

# add a bar for each category
for category in categories:
    marker = dict(color=color_map[category])
    if category == 'Academia and industry collaboration':
        marker = dict(
            color=color_map[category],
            pattern=dict(shape='/', fgcolor='#ffffff',
                         bgcolor=color_map[category], solidity=0.35, size=6)
        )

    fig.add_trace(go.Bar(
        x=df_wide.index,
        y=df_wide[category],
        name=category,
        marker=marker,
        customdata=custom,
        hovertemplate=make_tooltip(category)
    ))

fig.update_layout(
    barmode='stack',
    template='plotly_white',
    hovermode='closest',
    hoverlabel=dict(bgcolor='white', font_color='#222222', align='left'),
    title='Affiliation of researchers building notable AI systems by year of publication',
    yaxis_title='Number of notable AI systems',
    xaxis=dict(type='category', tickangle=0, nticks=10,
               minallowed=-0.5, maxallowed=len(df_wide) - 0.5),
    yaxis=dict(fixedrange=True),
    margin=dict(t=120),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0, y=1.0, xanchor='left', yanchor='bottom',
        pad=dict(b=8),
        buttons=[
            dict(label='Counts', method='relayout',
                 args=[{'barnorm': None,
                        'yaxis.title.text': 'Number of notable AI systems',
                        'yaxis.ticksuffix': '',
                        'yaxis.autorange': True,
                        'annotations[0].visible': True,
                        'annotations[1].visible': False}]),
            dict(label='Share (%)', method='relayout',
                 args=[{'barnorm': 'percent',
                        'yaxis.title.text': 'Share of notable AI systems (%)',
                        'yaxis.ticksuffix': '%',
                        'yaxis.autorange': True,
                        'annotations[0].visible': False,
                        'annotations[1].visible': True}])
        ]
    )]
)

#last year of data
last_year = df_wide.index.max()
industry_count = int(df_wide.loc[last_year, 'Industry'])
total_count = int(totals.loc[last_year])
industry_percentage = shares.loc[last_year, 'Industry']

# shown in Counts view: arrow points into the Industry block
fig.add_annotation(
    text=f'{last_year}: {industry_count} of {total_count} systems<br>came from industry',
    x=len(df_wide) - 0.6, y=industry_count / 2, xref='x', yref='y',
    ax=130, ay=0, showarrow=True, arrowhead=2,
    arrowcolor='#222222', arrowwidth=1.5,
    font=dict(size=12, color='#222222'), align='left',
    bgcolor='white', bordercolor='#999999', borderpad=4, visible=True
)

# shown in Share view: arrow points into the Industry block
fig.add_annotation(
    text=f'{last_year}: {industry_percentage:.1f}% of systems<br>came from industry',
    x=len(df_wide) - 0.6, y=industry_percentage / 2, xref='x', yref='y',
    ax=130, ay=0, showarrow=True, arrowhead=2,
    arrowcolor='#222222', arrowwidth=1.5,
    font=dict(size=12, color='#222222'), align='left',
    bgcolor='white', bordercolor='#999999', borderpad=4, visible=False
)

fig.show()